### Tools for generating and storing the chartbook

In [1]:
import os, shutil, subprocess, datetime, re, time
import pdfreader as pdf 
from pypdf import PdfReader
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

web_dir = Path(os.environ['WEBSITE_DIR']) if os.environ.get('WEBSITE_DIR') else None

os.chdir('../chartbook')

#### Run lualatex to generate chartbook PDF

In [2]:
# Run chartbook to generate pdf file
t0 = time.time()
runcb = subprocess.run(['lualatex', 'chartbook.tex'], stdout=subprocess.DEVNULL)
print(f'{runcb.returncode} — {time.time() - t0:.1f}s')

This is makeindex, version 2.18 [TeX Live 2026] (kpathsea + Thai support).
Scanning input file chartbook.idx....done (427 entries accepted, 0 rejected).
Sorting entries......done (3887 comparisons).
Generating output file chartbook.ind....done (368 lines written, 0 warnings).
Output written in chartbook.ind.
Transcript written in chartbook.ilg.


0 — 170.2s


#### Check for NaN (bugs in the code)

In [3]:
# nan represents "not a number" and indicates an error with the data generation
file = 'chartbook.pdf'

if runcb.returncode == 0:
    for i, p in enumerate(PdfReader(file).pages):
        if 'nan ' in p.extract_text():
            print(f'Found NaN; see page: {i+1}')

#### Move PDF to website folder

In [4]:
# Copy pdf file to personal website folder
if web_dir:
    shutil.copy('chartbook.pdf', web_dir.parent)

#### Check document length

In [5]:
curr_len = 185
file = open('chartbook.pdf', 'rb') 
if len([p for p in pdf.PDFDocument(file).pages()]) == curr_len:
    print('Page length correct')
else:
    print('ERROR! Incorrect page length')

Page length correct


#### Save Latest Date

In [6]:
dt = datetime.date.today().strftime('%B %-d, %Y')
if web_dir:
    with open(web_dir.parent / 'date.txt', 'w') as text_file:
        text_file.write(dt)

#### Update sitemap

In [7]:
if web_dir:
    sitemap = web_dir.parent / 'sitemap.xml'
    today = datetime.date.today().strftime('%Y-%m-%d')

    xml = open(sitemap).read()
    xml = re.sub(
        r'(chartbook\.html</loc>\s*<lastmod>)\d{4}-\d{2}-\d{2}',
        rf'\g<1>{today}', xml)
    open(sitemap, 'w').write(xml)

#### Update data sources JSON

In [8]:
result = subprocess.run(['python', '../notebooks/build_sources_json.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Wrote 47 releases to /home/brian/Documents/bdecon.github.io/files/chartbook_sources.json
Recent: Consumer Price Index, Treasury Statement, Fed Balance Sheet, Construction Spending, Consumer Expectations, Mortgage Rates, Initial Jobless Claims, Labor Productivity, State Unemployment, JOLTS

